# Toybox batch 1 — 60 questions, `openai/gpt-6.2-astra` (GPT-6 Luna and Muse Glimmer results already in the repo)

Same format and prompt as the single pilot question (notebook 53): 3 before/after demonstrations of 3 different
objects, 1 query image of a 4th object, 4 candidates, "Answer with exactly one letter".

| task | n | transformation | candidates |
|---|---|---|---|
| `R` rotation | 10 | 90° counter-clockwise, 90° clockwise or 180°, from recorded frames | 0 / 90 / 180 / 270° |
| `T` translation | 10 | left / right / up / down (crop-window shift = camera pan) | the 4 directions |
| `RT` rotation + translation | 10 | ±90° rotation (recorded) plus a shift | correct, rotation only, translation only, wrong rotation direction + shift |
| `M` mirror | 10 | left-right mirror (digital flip) about the object's own axis | correct, top-bottom mirror, rotate 180°, unchanged |
| `RM` rotation then mirror | 10 | ±90° rotation (recorded) then left-right mirror | correct, rotation only, mirror only, wrong order |
| `TM` translation + mirror | 10 | left-right mirror plus a shift | correct, mirror only, shift only, top-bottom mirror + shift |

The questions are already in the repository (`experiments/toybox_rotation_pilot/batch1/`). Answers live in
`batch1/answers.json` and are only read when scoring. One call per question. GPT-6 Luna and Muse Glimmer have already been run (results in `batch1/model_outputs/`).
This notebook calls `openai/gpt-6.2-astra` ($8 / $40 per 1M input / output tokens): cell 5 is a 2-question pilot that
prints the real token use and the projected cost of all 60; cell 6 runs the rest (estimated $5-15). Re-running
skips questions that already have a successful response.

Needs the Colab secrets `GITHUB_TOKEN` and `OPENROUTER_API_KEY` (never printed).


In [ ]:
# --- 2. Bootstrap: locate or clone the PRIVATE repository (credentials are never printed or stored) ---
import base64, importlib.util, os, pathlib, subprocess, sys

REPO_URL = "https://github.com/zhexuandliu/Decompositionality.git"
REPO_BRANCH = "main"
IN_COLAB = "google.colab" in sys.modules or "COLAB_RELEASE_TAG" in os.environ

def _find_repo():
    candidates = [pathlib.Path("/content/Decompositionality"), pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
    for p in candidates:
        if (p / "src" / "decompositionality").is_dir():
            return p.resolve()
    return None

REPO_DIR = _find_repo()
if REPO_DIR is None:
    token = None
    if IN_COLAB:
        try:
            from google.colab import userdata
            token = userdata.get("GITHUB_TOKEN")
        except Exception:
            token = None
    if not token:
        import getpass
        token = getpass.getpass("GitHub token with read access to the private repo (input hidden): ")
    # Pass the credential through environment-scoped git config (not the URL, not the command line,
    # not persisted in .git/config).
    _hdr = "AUTHORIZATION: basic " + base64.b64encode(f"x-access-token:{token}".encode()).decode()
    _env = dict(os.environ, GIT_TERMINAL_PROMPT="0", GIT_CONFIG_COUNT="1",
                GIT_CONFIG_KEY_0="http.https://github.com/.extraheader", GIT_CONFIG_VALUE_0=_hdr)
    REPO_DIR = pathlib.Path("/content/Decompositionality" if IN_COLAB else "Decompositionality").resolve()
    _r = subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, str(REPO_DIR)],
                        env=_env, capture_output=True, text=True)
    del token, _hdr, _env
    if _r.returncode != 0:
        raise RuntimeError("git clone failed (check the GITHUB_TOKEN secret / repo access):\n" + _r.stderr[-2000:])

if str(REPO_DIR / "src") not in sys.path:
    sys.path.insert(0, str(REPO_DIR / "src"))
for _mod in ("numpy", "pandas", "matplotlib", "torch"):
    if importlib.util.find_spec(_mod) is None:  # Colab already ships these; never reinstall torch/CUDA here
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", _mod], check=True)
REPO_COMMIT = subprocess.run(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
print("Repository:", REPO_DIR, "| commit:", REPO_COMMIT or "unknown")


In [ ]:
# --- 3. Update the repository (the question files were added recently) -------------------------------------------
import base64, subprocess
_tok = None
if IN_COLAB:
    from google.colab import userdata
    _tok = userdata.get("GITHUB_TOKEN")
_env = dict(os.environ, GIT_TERMINAL_PROMPT="0")
if _tok:
    _env.update(GIT_CONFIG_COUNT="1", GIT_CONFIG_KEY_0="http.https://github.com/.extraheader",
                GIT_CONFIG_VALUE_0="AUTHORIZATION: basic " + base64.b64encode(f"x-access-token:{_tok}".encode()).decode())
_r = subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", "main"], env=_env, capture_output=True, text=True)
del _tok, _env
print(_r.stdout[-500:] or _r.stderr[-500:])
PILOT = REPO_DIR / "experiments" / "toybox_rotation_pilot"
assert (PILOT / "batch1" / "answers.json").exists(), "batch files missing: pull failed?"
print("batch folder:", PILOT / "batch1")


In [ ]:
# --- 4. One example question (composites are for viewing only; they are not sent) -----------------------------
from IPython.display import Image as _Img, display
BATCH = PILOT / "batch1"
for qid in ["R00", "T00", "RT00", "M00", "RM00", "TM00"]:
    print(qid); display(_Img(filename=str(BATCH / qid / "composite.jpg"), width=900))


In [ ]:
# --- 5. Pilot: 2 questions with Astra, then the projected cost of all 60 ----------------------------------------------
if IN_COLAB and not os.environ.get("OPENROUTER_API_KEY"):
    from google.colab import userdata
    os.environ["OPENROUTER_API_KEY"] = userdata.get("OPENROUTER_API_KEY")      # never printed
import json
MODEL = "openai/gpt-6.2-astra"
MAX_TOKENS = 48000                     # generous cap so long reasoning is not truncated
PRICE_IN, PRICE_OUT = 8 / 1e6, 40 / 1e6
def run(limit=None):
    cmd = [sys.executable, str(PILOT / "eval_batch.py"), "--batch", str(BATCH), "--model", MODEL, "--workers", "8",
           "--max-tokens", str(MAX_TOKENS)] + (["--limit", str(limit)] if limit else [])
    r = subprocess.run(cmd, capture_output=True, text=True, cwd=str(PILOT))
    print(r.stdout[-6000:]); print(r.stderr[-3000:])
run(limit=2)
recs = [json.loads(l) for l in (BATCH / f"responses_{MODEL.replace('/', '_')}.jsonl").read_text().splitlines()]
ok = [r for r in recs if r.get("error") is None and r.get("usage")]
for r in ok:
    u = r["usage"]
    print(r["qid"], "answer:", repr(r["raw_response"]), "| prompt", u.get("prompt_tokens"), "completion", u.get("completion_tokens"),
          "| cost $", u.get("cost"), "| finish", r.get("finish_reason"))
if ok:
    pin = sum(r["usage"]["prompt_tokens"] for r in ok) / len(ok); pout = sum(r["usage"]["completion_tokens"] for r in ok) / len(ok)
    print(f"\\nprojected cost of 60 questions: ${60 * (pin * PRICE_IN + pout * PRICE_OUT):.2f}"
          f"  (mean {pin:.0f} input / {pout:.0f} output tokens per question)")


In [ ]:
# --- 6. Full run (resumable: the pilot questions are skipped) --------------------------------------------------------
run()


In [ ]:
# --- 6. Results (all models with responses) and download ------------------------------------------------------------
import pandas as pd, shutil, glob
sys.path.insert(0, str(PILOT))
from eval_batch import score
pd.set_option("display.width", 200); pd.set_option("display.max_colwidth", 70)
frames, seen = [], set()
for f in sorted(glob.glob(str(BATCH / "responses_*.jsonl")) + glob.glob(str(BATCH / "model_outputs" / "responses_*.jsonl"))):
    m = pathlib.Path(f).stem.replace("responses_", "")
    if m in seen:                       # a model's fresh responses (batch1/) take precedence over the saved copy
        continue
    seen.add(m)
    d = score(BATCH, pathlib.Path(f)); d["model"] = m; frames.append(d)
df = pd.concat(frames, ignore_index=True)
df["task"] = df.qid.str.rstrip("0123456789")
order = ["R", "T", "RT", "M", "RM", "TM"]
print("Accuracy by task and model:")
display(df.pivot_table(index="task", columns="model", values="correct").reindex(order).round(2))
print("\\nWhat was chosen when wrong:")
display(df[~df.correct].groupby(["model", "task", "chosen_kind"]).size().rename("n").reset_index())
print("\\nInvalid answers:", df.groupby("model").choice.apply(lambda s: int(s.isna().sum())).to_dict())
print("cost by model ($):", df.groupby("model").cost.sum().round(3).to_dict())
if IN_COLAB:
    from google.colab import files
    out = "/content/toybox_batch1_results"
    shutil.rmtree(out, ignore_errors=True); os.makedirs(out)
    for f in BATCH.glob("responses_*.jsonl"):
        shutil.copy(f, out)
    for f in BATCH.glob("scores_*.csv"):
        shutil.copy(f, out)
    shutil.make_archive(out, "zip", out)
    files.download(out + ".zip")
